# 01 — QLoRA Fine-tuning : Qwen3.5-2B sur GSM8K

**Objectif** : Fine-tuner Qwen3.5-2B pour le raisonnement mathématique chain-of-thought (CoT), en utilisant QLoRA via Unsloth sur une T4 Colab (16GB VRAM).

Pipeline complet :
```
fine-tune (ce notebook) → TurboQuant compression → évaluation before/after
```

**Stack** : Unsloth + QLoRA 4-bit + TRL SFTTrainer + GSM8K

---
## Architecture QLoRA

```
Qwen3.5-2B (gelé, 4-bit)      LoRA adapters (entraînables, fp16)
┌──────────────────┐           ┌────────────────────────┐
│  W (2048x2048)   │     +     │  A (2048xr)  r=16      │
│  quantifié 4-bit │           │  B (rx2048)            │
│  ~1.2GB total    │           │  ~65K params/couche    │
└──────────────────┘           └────────────────────────┘
   jamais mis à jour              seuls ces poids sont entraînés
```

## 1. Installation

Unsloth fournit :
- Chargement optimisé 4-bit (via bitsandbytes)
- Kernels CUDA custom pour QLoRA (2x plus rapide que PEFT vanilla)
- Intégration TRL pour le fine-tuning supervisé


In [ ]:
!nvidia-smi
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets trl transformers accelerate

## 1b. Google Drive — persistance

Les runtimes Colab sont éphémères : tout fichier écrit dans `/content/` disparaît à la déconnexion.
On monte Drive pour persister les adaptateurs LoRA entre les sessions.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Chemins persistants sur Drive
DRIVE_BASE = '/content/drive/MyDrive/efficient-llm-pipeline'
LORA_PATH  = f'{DRIVE_BASE}/models/qwen-gsm8k-lora'
CKPT_PATH  = f'{DRIVE_BASE}/checkpoints'

os.makedirs(LORA_PATH, exist_ok=True)
os.makedirs(CKPT_PATH, exist_ok=True)

print(f'Drive monté. Adaptateurs → {LORA_PATH}')

## 2. Chargement du modèle en 4-bit

| Paramètre | Valeur | Explication |
|---|---|---|
| `load_in_4bit` | `True` | Quantification NF4 — meilleure précision que int4 pour les LLMs |
| `max_seq_length` | `2048` | GSM8K solutions ~200-400 tokens, marge incluse |
| `dtype` | `None` | Auto-détection bfloat16/float16 |

**NF4 vs INT4** : Les poids LLM suivent une distribution normale. NF4 optimise ses 16 niveaux pour cette distribution → meilleure précision à même taille mémoire.


In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen3.5-2B"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print(f"Modèle chargé : {MODEL_NAME}")
print(f"VRAM utilisée : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. Configuration LoRA

| Param | Valeur | Rôle |
|---|---|---|
| `r` | 16 | Rang des matrices A et B |
| `lora_alpha` | 32 | Scaling = alpha/r = 2.0 |
| `target_modules` | q/k/v/o + FFN | Couches attention + MLP |
| `gradient_checkpointing` | unsloth | -40% VRAM, léger overhead calcul |


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=14,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Paramètres entraînables : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4. Dataset GSM8K + formatage chat template

Chaque exemple est converti en conversation ChatML :
```
system    → instruction CoT step-by-step
user      → la question
assistant → solution + #### <réponse>
```
Qwen3.5 a été pré-entraîné avec ce format — sans lui, le modèle ne sait pas où commencer sa réponse.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")
print(f"Train : {len(dataset['train'])} | Test : {len(dataset['test'])}")

In [ ]:
SYSTEM_PROMPT = (
    "Tu es un assistant mathématique expert. "
    "Décompose chaque problème étape par étape en montrant tous les calculs. "
    "Termine toujours par '#### <réponse>' sur la dernière ligne."
)

def format_example(example):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_dataset = dataset["train"].map(format_example, remove_columns=["question", "answer"])
eval_dataset  = dataset["test"].map(format_example,  remove_columns=["question", "answer"])
print(train_dataset[0]["text"][:400])

## 5. Entraînement — SFTTrainer

| Paramètre | Valeur | Pourquoi |
|---|---|---|
| `per_device_train_batch_size` | 2 | T4 16GB limite |
| `gradient_accumulation_steps` | 4 | Batch effectif = 8 sans OOM |
| `max_steps` | 200 | ~30min T4, suffisant pour converger |
| `learning_rate` | 2e-4 | Standard LoRA |
| `optim` | adamw_8bit | ~3GB VRAM économisés vs Adam 32-bit |


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=1,   # réduit de 2 : Qwen3.5 hybrid arch consomme plus
        gradient_accumulation_steps=8,   # augmenté de 4 : batch effectif = 8 inchangé
        warmup_steps=20,
        max_steps=200,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=CKPT_PATH,
        report_to="none",
    ),
)

In [ ]:
print("Fine-tuning QLoRA (~30min sur T4)...")
trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3)
print(f"Temps : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"VRAM peak : {used_memory} / {max_memory} GB")
print(f"Loss finale : {trainer_stats.metrics['train_loss']:.4f}")

## 6. Sauvegarde des adaptateurs LoRA

On sauvegarde uniquement les adaptateurs (~50MB), pas le modèle complet (~8GB).
Ce sont ces adaptateurs qu'on charge dans `02_turboquant.ipynb`.


In [ ]:
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)

size_mb = sum(os.path.getsize(os.path.join(LORA_PATH, f)) for f in os.listdir(LORA_PATH)) / 1e6
print(f"Adaptateurs sauvegardés : {LORA_PATH} ({size_mb:.1f} MB)")

## 7. Test d'inférence

3 exemples du test set (jamais vus) pour vérifier que le modèle produit bien du CoT.


In [ ]:
FastLanguageModel.for_inference(model)

def solve(question):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [{"type": "text", "text": question}]},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

for i, ex in enumerate(dataset["test"].select(range(3))):
    print(f"\n{'='*60}\nQ{i+1}: {ex['question']}\n\nAttendu:\n{ex['answer']}\n\nModèle:")
    print(solve(ex['question']))

## Récapitulatif

| Étape | Technique | Impact mémoire |
|---|---|---|
| Chargement | QLoRA NF4 4-bit | ~2GB au lieu de ~8GB |
| Adaptation | LoRA r=16 sur 7 modules | ~0.5% des params entraînables |
| Entraînement | Gradient accumulation + adamw_8bit | Batch effectif 8 sans OOM |
| Sauvegarde | Adaptateurs LoRA uniquement | ~50MB au lieu de ~8GB |

**Suite → `02_turboquant.ipynb`** : implémentation TurboQuant KV cache compression depuis le paper (mars 2026).

---
*[efficient-llm-pipeline](https://github.com/YanissAmz/efficient-llm-pipeline)*
